# Modelo de clasificación con Random Forest

**Proyecto:** Detección de estudiantes con depresión  
**Dataset:** `Student Depression Dataset.csv`

Este notebook prueba un enfoque de **clasificación** usando **Random Forest**, ya que la variable objetivo `Depression` tiene valores binarios: `0` y `1`.

- `0` = No presenta depresión
- `1` = Presenta depresión

A diferencia de la regresión polinómica, aquí el modelo está diseñado para responder directamente **sí/no**.

## Contexto humano del proyecto

Aunque este notebook utiliza un modelo de machine learning, el objetivo no es reemplazar el criterio de un profesional de la salud mental.  
La depresión en estudiantes es un fenómeno complejo que puede estar relacionado con presión académica, sueño insuficiente, estrés financiero, satisfacción con los estudios, antecedentes familiares y otros factores personales.

Este proyecto puede entenderse como una **herramienta de apoyo para la detección temprana**, no como un sistema de diagnóstico automático.

En un contexto cotidiano, podría ser útil para:

- **Psicólogos escolares o universitarios:** como una herramienta preliminar para priorizar entrevistas de seguimiento.
- **Orientadores educativos:** para identificar grupos de estudiantes que podrían necesitar acompañamiento.
- **Departamentos de bienestar estudiantil:** para diseñar campañas preventivas sobre sueño, estrés, alimentación y salud emocional.
- **Instituciones educativas:** para observar patrones generales de riesgo y fortalecer programas de apoyo.

La idea principal es usar el modelo para **abrir una conversación humana**, no para etiquetar definitivamente a una persona.

## Objetivo aplicado

El objetivo práctico del modelo es estimar si un estudiante tiene mayor probabilidad de presentar depresión a partir de variables relacionadas con su vida académica, hábitos y contexto personal.

Una posible pregunta de aplicación sería:

> Si una institución educativa cuenta con información básica de bienestar estudiantil, ¿puede identificar de manera preventiva a estudiantes que podrían requerir acompañamiento psicológico?

Desde esta perspectiva, el modelo no debe verse como una respuesta final, sino como un **filtro de apoyo** para canalizar recursos humanos de forma más oportuna.

## Paso 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

## Paso 2. Cargar el dataset

El archivo debe estar en la misma carpeta del notebook.

In [ ]:
df = pd.read_csv("Student Depression Dataset.csv")

print("Tamaño del dataset:", df.shape)
df.head()

## Paso 3. Revisar columnas y valores nulos

In [ ]:
print("Columnas del dataset:")
print(df.columns.tolist())

print("
Valores nulos por columna:")
print(df.isnull().sum())

## Paso 4. Revisar valores únicos de columnas categóricas

Esto permite confirmar cómo vienen escritas las categorías antes de convertirlas a números.

In [ ]:
columnas_categoricas_revision = [
    "Gender",
    "Sleep Duration",
    "Dietary Habits",
    "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness"
]

for col in columnas_categoricas_revision:
    print(f"
{col}:")
    print(df[col].dropna().unique())

## Paso 5. Limpieza y transformación de datos

Random Forest necesita datos numéricos. Por eso convertimos las columnas categóricas principales a valores numéricos.

In [ ]:
df_limpio = df.copy()

# Gender
map_gender = {
    "Male": 0,
    "Female": 1
}
df_limpio["Gender"] = df_limpio["Gender"].map(map_gender)

# Sleep Duration
map_sleep = {
    "Less than 5 hours": 4.0,
    "5-6 hours": 5.5,
    "7-8 hours": 7.5,
    "More than 8 hours": 9.0,
    "Others": np.nan
}
df_limpio["Sleep Duration"] = df_limpio["Sleep Duration"].map(map_sleep)

# Dietary Habits
map_diet = {
    "Unhealthy": 0,
    "Moderate": 1,
    "Healthy": 2,
    "Others": np.nan
}
df_limpio["Dietary Habits"] = df_limpio["Dietary Habits"].map(map_diet)

# Suicidal thoughts
map_suicidal = {
    "No": 0,
    "Yes": 1
}
df_limpio["Have you ever had suicidal thoughts ?"] = df_limpio["Have you ever had suicidal thoughts ?"].map(map_suicidal)

# Family history
map_family = {
    "No": 0,
    "Yes": 1
}
df_limpio["Family History of Mental Illness"] = df_limpio["Family History of Mental Illness"].map(map_family)

## Paso 6. Seleccionar variables para el modelo

Se eliminan columnas muy categóricas como `City`, `Profession` y `Degree` para mantener el modelo más simple y entendible.

Estas variables se pueden agregar después con técnicas de codificación, pero para este primer Random Forest usaremos variables numéricas y categorías simples ya transformadas.

## Consideraciones éticas antes del modelado

Trabajar con salud mental requiere especial cuidado. Por eso, este proyecto debe interpretarse bajo estas condiciones:

1. **No es un diagnóstico clínico.**  
   El modelo clasifica patrones dentro de un dataset, pero no puede determinar por sí mismo si una persona tiene depresión.

2. **La decisión final debe ser humana.**  
   Un psicólogo, orientador o profesional capacitado debe revisar cualquier caso sensible.

3. **La privacidad es fundamental.**  
   Los datos de estudiantes deben manejarse de forma anónima, segura y con consentimiento informado.

4. **Evitar etiquetas dañinas.**  
   Una predicción alta no significa que el estudiante “sea depresivo”; significa que podría beneficiarse de una revisión o conversación de apoyo.

5. **Atención prioritaria ante señales de riesgo.**  
   Si una persona expresa ideas suicidas actuales, eso debe atenderse de inmediato por canales profesionales, independientemente de lo que prediga el modelo.

In [ ]:
columnas_modelo = [
    "Age",
    "Gender",
    "Academic Pressure",
    "Work Pressure",
    "CGPA",
    "Study Satisfaction",
    "Job Satisfaction",
    "Sleep Duration",
    "Dietary Habits",
    "Have you ever had suicidal thoughts ?",
    "Work/Study Hours",
    "Financial Stress",
    "Family History of Mental Illness"
]

objetivo = "Depression

## Paso 7. Convertir variables a numéricas y eliminar nulos

In [ ]:
for col in columnas_modelo + [objetivo]:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce")

print("Nulos antes de limpiar:")
print(df_limpio[columnas_modelo + [objetivo]].isnull().sum())

df_modelo = df_limpio.dropna(subset=columnas_modelo + [objetivo]).copy()

print("
Tamaño original:", df.shape)
print("Tamaño después de limpieza:", df_modelo.shape)

df_modelo[columnas_modelo + [objetivo]].head()

## Paso 8. Separar X y y

In [ ]:
X = df_modelo[columnas_modelo]
y = df_modelo[objetivo]

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)

print("
Distribución de la variable objetivo:")
print(y.value_counts())
print("
Distribución porcentual:")
print(y.value_counts(normalize=True) * 100)

## Paso 9. Crear el modelo Random Forest

Parámetros principales:

- `n_estimators=100`: cantidad de árboles del bosque.
- `max_depth=10`: limita la profundidad para reducir sobreajuste.
- `class_weight="balanced"`: ayuda si hay desbalance entre clases.
- `random_state=42`: permite reproducibilidad.

In [ ]:
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

## Paso 10. Validación cruzada con 5 folds

Aquí evaluamos el modelo en 5 escenarios distintos.

A diferencia del KFold normal, usamos `StratifiedKFold`, que mantiene una proporción parecida de `0` y `1` en cada fold.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=False)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

resultados_cv = cross_validate(
    modelo_rf,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

resultados_folds = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "Accuracy": resultados_cv["test_accuracy"],
    "Precision": resultados_cv["test_precision"],
    "Recall": resultados_cv["test_recall"],
    "F1": resultados_cv["test_f1"],
    "ROC_AUC": resultados_cv["test_roc_auc"]
})

resultados_folds

## Paso 11. Promedios de evaluación

In [ ]:
promedios = pd.DataFrame({
    "Metrica": ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"],
    "Promedio": [
        resultados_folds["Accuracy"].mean(),
        resultados_folds["Precision"].mean(),
        resultados_folds["Recall"].mean(),
        resultados_folds["F1"].mean(),
        resultados_folds["ROC_AUC"].mean()
    ]
})

promedios

## Interpretación humana de las métricas

En un problema de salud mental, no todas las métricas tienen el mismo peso.

- **Accuracy** mide cuántas predicciones totales fueron correctas, pero puede ocultar errores importantes.
- **Precision** indica qué tan confiables son las alertas positivas del modelo.
- **Recall** es especialmente importante porque mide cuántos casos reales de depresión fueron detectados.
- **F1-score** resume el equilibrio entre precision y recall.
- **ROC AUC** evalúa qué tan bien separa el modelo entre estudiantes con y sin depresión.

En este contexto, un falso negativo puede ser delicado:

> Falso negativo: el modelo predice “sin depresión”, pero el estudiante sí presentaba depresión.

Por eso, si el proyecto se aplicara en una institución, sería preferible usar el modelo como una herramienta de **acompañamiento preventivo**, no como un filtro excluyente.

## Paso 12. Gráficas de métricas por fold

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(resultados_folds["Fold"], resultados_folds["Accuracy"], marker="o", label="Accuracy")
plt.plot(resultados_folds["Fold"], resultados_folds["F1"], marker="o", label="F1")
plt.plot(resultados_folds["Fold"], resultados_folds["ROC_AUC"], marker="o", label="ROC AUC")
plt.title("Métricas por fold - Random Forest")
plt.xlabel("Fold")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

## Paso 13. Separar datos en entrenamiento y prueba final

Este bloque sirve para mostrar matriz de confusión y reporte de clasificación con una división 80/20.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

modelo_rf.fit(X_train, y_train)
y_pred = modelo_rf.predict(X_test)
y_prob = modelo_rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

## Paso 14. Matriz de confusión

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["No depresión", "Depresión"]
)

disp.plot()
plt.title("Matriz de confusión - Random Forest")
plt.show()

## Lectura práctica de la matriz de confusión

La matriz de confusión debe leerse con cuidado:

- **Verdadero positivo:** el modelo detecta correctamente a un estudiante con depresión.
- **Verdadero negativo:** el modelo identifica correctamente a un estudiante sin depresión.
- **Falso positivo:** el modelo alerta posible depresión, pero el estudiante no pertenece a esa clase en el dataset.
- **Falso negativo:** el modelo no alerta, aunque el estudiante sí pertenece a la clase de depresión.

En un entorno escolar, un falso positivo puede significar una entrevista adicional.  
Un falso negativo puede significar que un estudiante que necesitaba apoyo no fue priorizado.

Por eso, en salud mental suele ser importante no enfocarse solo en el accuracy, sino también revisar **recall**, **F1-score** y el contexto humano del uso del modelo.

## Paso 15. Reporte de clasificación

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["No depresión", "Depresión"]
))

## Paso 16. Curva ROC

In [ ]:
RocCurveDisplay.from_estimator(modelo_rf, X_test, y_test)
plt.title("Curva ROC - Random Forest")
plt.grid(True)
plt.show()

## Paso 17. Importancia de variables

Random Forest permite identificar qué variables fueron más importantes para decidir la clasificación.

In [ ]:
importancias = pd.DataFrame({
    "Variable": columnas_modelo,
    "Importancia": modelo_rf.feature_importances_
}).sort_values(by="Importancia", ascending=False)

importancias

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(importancias["Variable"], importancias["Importancia"])
plt.title("Importancia de variables - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.gca().invert_yaxis()
plt.grid(True)
plt.show()

## Interpretación humana de la importancia de variables

La gráfica de importancia de variables ayuda a entender qué factores usó más el modelo para clasificar.

Sin embargo, una variable importante no debe interpretarse automáticamente como una causa directa.  
Por ejemplo, si `Academic Pressure` o `Sleep Duration` aparecen con alta importancia, eso no significa que por sí solas causen depresión. Significa que, dentro de este dataset, ayudaron al modelo a distinguir patrones entre estudiantes.

Esta lectura puede servir a una institución para reflexionar sobre preguntas como:

- ¿La presión académica está relacionada con mayor riesgo emocional?
- ¿Los estudiantes con menos horas de sueño requieren estrategias de bienestar?
- ¿El estrés financiero aparece como un factor relevante?
- ¿La satisfacción con los estudios podría funcionar como señal temprana?

El valor del modelo no está únicamente en predecir, sino también en ayudar a identificar áreas donde se pueden implementar acciones preventivas.

## Paso 18. Entrenar modelo final con todos los datos

Después de evaluar el modelo, se entrena una versión final usando todos los datos disponibles.

In [ ]:
modelo_final_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

modelo_final_rf.fit(X, y)

## Paso 19. Insertar un dato nuevo y predecir

Debes respetar las mismas columnas usadas en `X`.

In [ ]:
nuevo_dato = pd.DataFrame([{
    "Age": 21,
    "Gender": 0,
    "Academic Pressure": 4,
    "Work Pressure": 0,
    "CGPA": 7.8,
    "Study Satisfaction": 3,
    "Job Satisfaction": 0,
    "Sleep Duration": 5.5,
    "Dietary Habits": 1,
    "Have you ever had suicidal thoughts ?": 0,
    "Work/Study Hours": 8,
    "Financial Stress": 3,
    "Family History of Mental Illness": 1
}])

prediccion_clase = modelo_final_rf.predict(nuevo_dato)[0]
prediccion_probabilidad = modelo_final_rf.predict_proba(nuevo_dato)[0]

print("Clase predicha:", prediccion_clase)
print("Probabilidad No depresión:", prediccion_probabilidad[0])
print("Probabilidad Depresión:", prediccion_probabilidad[1])

if prediccion_clase == 1:
    print("Resultado: El modelo clasifica al estudiante como CON depresión.")
else:
    print("Resultado: El modelo clasifica al estudiante como SIN depresión.")

## Paso 20. Probar varios casos nuevos

In [ ]:
nuevos_datos = pd.DataFrame([
    {
        "Age": 20,
        "Gender": 0,
        "Academic Pressure": 5,
        "Work Pressure": 0,
        "CGPA": 6.5,
        "Study Satisfaction": 1,
        "Job Satisfaction": 0,
        "Sleep Duration": 4.0,
        "Dietary Habits": 0,
        "Have you ever had suicidal thoughts ?": 1,
        "Work/Study Hours": 12,
        "Financial Stress": 5,
        "Family History of Mental Illness": 1
    },
    {
        "Age": 22,
        "Gender": 1,
        "Academic Pressure": 2,
        "Work Pressure": 0,
        "CGPA": 8.7,
        "Study Satisfaction": 4,
        "Job Satisfaction": 0,
        "Sleep Duration": 7.5,
        "Dietary Habits": 2,
        "Have you ever had suicidal thoughts ?": 0,
        "Work/Study Hours": 5,
        "Financial Stress": 1,
        "Family History of Mental Illness": 0
    }
])

pred_clases = modelo_final_rf.predict(nuevos_datos)
pred_probs = modelo_final_rf.predict_proba(nuevos_datos)[:, 1]

resultado_predicciones = nuevos_datos.copy()
resultado_predicciones["Prediccion"] = pred_clases
resultado_predicciones["Probabilidad_Depresion"] = pred_probs
resultado_predicciones["Interpretacion"] = np.where(
    resultado_predicciones["Prediccion"] == 1,
    "Con depresión",
    "Sin depresión"
)

resultado_predicciones

## Paso 21. Semáforo de apoyo para interpretar predicciones

Además de clasificar `0` o `1`, podemos usar la probabilidad estimada por el modelo para crear un semáforo de apoyo.

Este semáforo no diagnostica.  
Su propósito es traducir el resultado del modelo a una recomendación más humana y entendible para un orientador, psicólogo o tutor académico.

Ejemplo de interpretación:

- **Riesgo bajo:** seguimiento preventivo general.
- **Riesgo moderado:** sugerir conversación de orientación o revisión de bienestar.
- **Riesgo alto:** priorizar contacto con personal capacitado.
- **Atención inmediata:** si el estudiante reporta pensamientos suicidas, se debe activar apoyo profesional sin depender del puntaje del modelo.

In [ ]:
def clasificar_semaforo(probabilidad, pensamientos_suicidas):
    if pensamientos_suicidas == 1:
        return "Atención prioritaria", (
            "El estudiante reporta pensamientos suicidas. "
            "Debe ser canalizado de inmediato con personal capacitado, "
            "independientemente de la probabilidad calculada por el modelo."
        )

    if probabilidad < 0.33:
        return "Riesgo bajo", (
            "Seguimiento preventivo general. "
            "Se pueden reforzar hábitos de sueño, alimentación, manejo de estrés "
            "y canales de apoyo disponibles."
        )

    elif probabilidad < 0.66:
        return "Riesgo moderado", (
            "Conviene sugerir una conversación con orientación, tutoría o bienestar estudiantil. "
            "No implica diagnóstico, pero sí amerita acompañamiento preventivo."
        )

    else:
        return "Riesgo alto", (
            "Se recomienda priorizar una revisión por parte de un psicólogo, orientador "
            "o profesional capacitado. El modelo solo indica una señal de alerta."
        )


resultado_predicciones_humano = resultado_predicciones.copy()

niveles = []
recomendaciones = []

for _, fila in resultado_predicciones_humano.iterrows():
    nivel, recomendacion = clasificar_semaforo(
        fila["Probabilidad_Depresion"],
        fila["Have you ever had suicidal thoughts ?"]
    )

    niveles.append(nivel)
    recomendaciones.append(recomendacion)

resultado_predicciones_humano["Semaforo_Apoyo"] = niveles
resultado_predicciones_humano["Recomendacion_Humana"] = recomendaciones

resultado_predicciones_humano

## Aplicación en la vida cotidiana

Una posible forma de uso sería dentro de un programa de bienestar estudiantil.

Por ejemplo, una universidad podría aplicar periódicamente cuestionarios breves sobre presión académica, sueño, estrés financiero y satisfacción con los estudios. Con esa información, el modelo podría ayudar a identificar estudiantes que podrían beneficiarse de una conversación de apoyo.

El flujo de uso podría ser:

```text
Cuestionario de bienestar
→ Modelo de clasificación
→ Semáforo de apoyo
→ Revisión humana
→ Canalización o seguimiento
```

Esto permitiría que psicólogos y orientadores no trabajen únicamente de forma reactiva, es decir, cuando el problema ya es muy evidente, sino también de manera preventiva.

La parte más importante no es la predicción aislada, sino la posibilidad de intervenir antes:

- contactar al estudiante;
- ofrecer recursos de apoyo;
- reducir carga o presión académica cuando sea posible;
- sugerir hábitos de sueño y alimentación;
- orientar hacia ayuda profesional si hay señales sensibles.

## Conclusión final humanizada

Este proyecto muestra cómo un modelo Random Forest puede utilizarse para clasificar estudiantes según la presencia o ausencia de depresión dentro de un dataset.

Desde el punto de vista técnico, el modelo permite evaluar métricas como accuracy, precision, recall, F1-score y ROC AUC. También permite observar qué variables influyen más en la clasificación.

Desde el punto de vista humano, el valor del proyecto está en su posible uso como **herramienta de prevención y acompañamiento**. El modelo puede ayudar a detectar patrones de riesgo y apoyar la toma de decisiones en contextos escolares o universitarios, pero nunca debe sustituir una evaluación profesional.

Una predicción alta debe interpretarse como una señal para acercarse con empatía, escuchar al estudiante y ofrecer ayuda.  
Una predicción baja tampoco debe usarse para ignorar el malestar de una persona si existen señales visibles o reportes directos.

En resumen:

```text
El modelo no diagnostica.
El modelo orienta.
La decisión final debe ser humana.
```

Este tipo de herramienta puede ser valiosa si se utiliza con responsabilidad, privacidad, sensibilidad y acompañamiento profesional.